# Agent 工作台工程：为什么有能力的模型依然失败

一个有能力的模型是不够的。可靠的agents需要一个工作台：指令、状态、范围、反馈、验证、审查以及交接。没有这些东西即使时前沿的模型仍然产出不安全的工作。

## 问题描述

你讲一个前沿的模型丢到一个真实的仓库，然后让它加上输入验证。它打开了四个文件，写下了似是而非的代码，然后宣告成功、并停止运行。你跑测试的时候，两个失败了，第三个是完全与验证无关的文件却没修改了。并且这里没有任务关于agent假设了什么、尝试了什么、以及什么没做完的记录。

模型犯的不是Python的错，而是关于任务的。它不知道什么意味着完成，哪里被允许做写入操作、哪些测试是可信的、或者下一个会话是否应该被拾起。

这不是模型的问题，而是工作台的问题。围绕在agent周围的这些层就是将单次生成推向可靠的、长久的工程所需要的零件。

## 基本概念

一个工作台是一次任务中包裹在模型周围的操作环境。有7个内容：
|内容|包含什么|缺失会导致什么问题|
|---|---|---|
|指令|启动规则、禁止行为、终止定义|agents在猜测含义|
|状态|当前任务、动到的文件、拦截、下个行动|每个会话都从零开始|
|范围|允许/禁止的文件、接受标准|编辑了不相关的内容|
|返回|将真实命令输入捕获回循环中|agents在错误时却宣称成功|
|验证|测试、冒烟、范围检查|似是而非|
|审查|从另一个角度判断是否通过|自己给自己打分|
|交接|改了什么，为什么、留下了什么|下一轮会话需要重新挖掘|

工作台与模型无关。你可以更改模型但是保留上层。但是不能期望修改了上层还要保持稳定性。

关注状态文件而不是会话历史，聊天是可变的，仓库才是记录系统。

### 工作台和提示词工程

提示词告诉模型你这轮想干嘛。工作台告诉模型怎么跨轮次和会话进行工作。大多数agent失败是披着提示词工程皮的工程台失败。

### 工程台和框架

框架给你是运行时（LangGraph，AutoGen，Agents SDK）。工作台给的是一个任务能在运行时工作的地方。两者你都需要。

### 从原语推理，而非依赖方的分类

现在有大量的“harness engineering”。它们对于harness的边界在哪、包含哪些内容、使用哪些词汇有很大的争议。我们不需要选一方站队。

暂时抛开agent的标签。一次agent调用就是跨时间、过程和机器的调用。你需要任何生产系统上相同的原语来使其变得可靠。

|原语|是什么|为agent带来了什么|
|---|---|---|
|Function|具型的处理器，有输入和输出|工具调用、规则检查、验证、模型调用|
|Worker|长周期的,包含一到多个函数的进程|建造者、审计者、验证者、MCP服务|
|Trigger|触发函数的事件源|agent循环节点、HTTP请求、队列信息、时程、钩子|
|Runtime|决定跑什么、在哪里跑、时间和资源限制是什么|Claude Code的process, LangGraph 的runtime，worker容器|
|HTTP/RPC|caller和worker的纽带|工具调用协议，MCP请求，模型API|
|Queue|trigger和worker的持久缓存|任务板、反馈日志、审查信箱|
|Session persistence|崩溃、重启、模型切换仍能保留的状态|检查点、KV存储、仓库本身|
|Authorization policy|谁能在哪个范围调用哪个函数|允许/禁止文件、许可边界、MCP能力列表|

将7个工作台表层映射到原语：
- 指令。 政策和函数的元数据。
- 状态。 会话持久化
- 范围。 每个任务的授权策略。
- 反馈。 写入队列的调用日志。
- 验证。 本身是函数
- 审查。 独立的只读worker，或者带写审查报告的能力。
- 交接。 会话结束时trigger发出的持久记录。

agent循环本身就是一个消费事件（用户消息，工具结果，时间戳）的worker，调用工具、写记录、然后触发triggers。没什么神秘的。

### 碎碎念

当你在其他地方听到“harness engineering”的时候，翻译到原语。Prompts和Rules是策略和函数。 Scaffolding是运行时。Guardrails 是授权和验证。Hooks是触发器。Memory时会话持久。Ralph Loop的重新入队。Subagents是workders。Sandbox是计算面。词汇在变，但是工程没有。

### 主流 harness 设计 → 原语

各家把同一套东西叫成不同名字。下表把常见产品/框架的设计面，翻译回上一节的原语。读别人的架构文档时，先做这层映射，再决定要不要抄他们的名词。

| 产品 / 框架 | 他们怎么说 | 翻译成原语 |
|---|---|---|
| Claude Code / Agent SDK | system prompt、CLAUDE.md、hooks、subagents、session store、MCP、permissions | 指令=策略元数据；hooks=`Trigger`；subagent=`Worker`；session=`Session persistence`；MCP/`tool`=`HTTP/RPC` + `Function`；permissions=`Authorization policy` |
| Cursor Agent | rules、skills、MCP、checkpoints、mode（Agent/Ask）、terminal sandbox | rules/skills=策略 + 函数元数据；checkpoint=`Session persistence`；sandbox=Runtime 的计算面；MCP=`HTTP/RPC` |
| OpenAI Agents SDK | handoffs、guardrails、tracing、sessions、tools | handoff=`Trigger`（点对点交接事件）；guardrail=`Authorization` + 验证 `Function`；session=持久化；tool=`Function` |
| LangGraph | StateGraph、node、edge、checkpointer、interrupt/HITL、subgraph | node=`Function`；graph runtime=`Runtime`；checkpointer=`Session persistence`；interrupt=`Trigger`（人在环）；subgraph=嵌套 `Worker` |
| AutoGen / AG2 | GroupChat、Speaker selection、message bus | speaker 选择=`Runtime` 调度；message=`Queue` 上的事件；每个 agent=`Worker` |
| CrewAI | Crew、Process（sequential/hierarchical）、Task、Agent | Process=`Runtime` 拓扑；Agent=`Worker`；Task=带授权范围的工作单元 |
| Aider / SWE-agent / OpenHands | repo map、edit format、test command loop、sandbox | repo map=状态投影；edit/test loop=`Worker` 消费 `Trigger`；sandbox=`Runtime` |
| MCP 生态 | tools / resources / prompts、client-server | tool/resource=`Function`；client↔server=`HTTP/RPC`；capability 列表=`Authorization policy` |

再压一层，几乎所有「harness 架构图」都能拆成同一句话：

1. **Trigger** 叫醒一个 **Worker**（用户消息、钩子、cron、队列消息、handoff）。
2. **Worker** 在某个 **Runtime** 里按 **Authorization** 调一组 **Function**（模型、工具、验证、审查）。
3. 调用经 **HTTP/RPC** 出去；结果与副作用写入 **Queue** / **Session persistence**。
4. 下一轮 Trigger 再读持久状态，而不是重读聊天窗口。

对照时别被商标带走：

- 「记忆 / Memory」→ 问它是会话窗口还是 `Session persistence`（跨崩溃）。
- 「护栏 / Guardrails」→ 拆成授权（能不能调）和验证（调完对不对）。
- 「循环 / Loop / Ralph」→ `Worker` + `Queue` 重新入队，不是神秘控制论。
- 「多智能体」→ 多个 `Worker` + 谁触发谁；拓扑是 Runtime 策略，不是新原语。

工作台的 7 层（指令…交接）是**任务侧装配**；上表是**业界侧叫法**。两边都映射到同一套原语，你就不必在 Claude / Cursor / LangGraph 的词汇之间挑边站队。